# 10 — Embedding signals into states

To *do* quantum optimal transport on a signal, you first face a modelling choice that decides everything downstream: **how does a signal become a quantum state?** This notebook lays out the embedding menu and, for each option, names exactly what structure of the signal it keeps — and what it throws away.

**Prerequisites:** `09_the_qot_zoo`.

**What you'll be able to do**
- Explain why the embedding is a **modelling decision**, not a technicality: the QOT objects are fixed, and the answer they return depends entirely on the state you hand them.
- Build the four embeddings of `qot_course.quantum_ot` on a toy phase-and-amplitude signal — the **phase qubit**, the **amplitude–phase** qubit, the **covariance** density, and the **multi-frequency** qutrit — and confirm each is a valid quantum state.
- Say **what each embedding preserves**: the phase qubit keeps only the first circular moment of the phase (exactly what PLV sees); amplitude–phase additionally tracks band power; covariance summarises phase dispersion; multi-frequency keeps the first *and* second circular moments.
- Show, on a clean toy, that two phase-difference signals with the **same first circular moment** but **different second moment** give the naive embedding the *same* joint coherence (the PLV-level quantity) yet give the multi-frequency embedding a *different* second-moment coherence — the structural reason a richer embedding can see what PLV cannot.

You have spent two modules building the transport machinery and one notebook reconciling it. Today you build the bridge from that machinery to real signals — and you will see that the bridge itself carries a scientific decision. Choose the embedding well and the geometry can see the structure that matters; choose it carelessly and you have rebuilt a classical baseline in quantum clothing. That choice is the whole subject here, and by the end you will make it deliberately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import phase_qubit_state, joint_density_matrix, plv
from qot_course.quantum_ot.embeddings import (
    amplitude_phase_state,
    covariance_density,
    multifreq_state,
    joint_density_from_states,
)
from qot_course.infotheory.quantum import quantum_mutual_information

np.random.seed(42)
viz.use_course_style()

## 1. The embedding is a modelling decision

In `09` you put the three quantum optimal-transport objects on one bench and pinned down exactly how they relate. Those objects are now *fixed*: the coupling SDP, the entropic plan, the Bures bridge — each takes a quantum state (or a pair of them) and returns a number. Nothing about them is going to change when we point them at data.

But a recorded signal — an EEG trace, an oscillator's phase — is **not** a quantum state. It is a sequence of real numbers. So before any QOT object can read it, we must decide how the signal *becomes* a state. That decision is the entire content of this notebook, and it is worth saying plainly why it carries so much weight:

> The QOT objects are fixed. The answer they give depends on the state you hand them. **The embedding is where the modelling lives.**

An embedding is a lossy summary. It cannot keep everything about a signal in a small density matrix, so it *chooses* what to keep. Two different embeddings of the same signal are two different states, and the same transport measure will generally return two different numbers for them — not because the measure changed, but because you asked it about different objects. This is not a flaw to be engineered away; it is the place where your scientific judgement enters. The rest of the notebook makes that judgement concrete: we survey the menu, name what each option preserves, and then show — on a deliberately simple toy — how the choice can decide whether the geometry sees a piece of structure at all.

## 2. The menu — four ways to turn a signal into a state

Here is the menu the course offers, built on one toy signal so the contrasts are concrete. The signal is a phase $\theta(t)$ that drifts in time, carrying an amplitude envelope $a(t) \in [0, 1]$ alongside it — the two quantities a band-limited neural signal hands you after a Hilbert transform (instantaneous phase and instantaneous amplitude).

The four embeddings, and what each keeps:

- **Phase qubit** (`phase_qubit_state`, the naive baseline): $\;(\lvert 0\rangle + e^{i\theta}\lvert 1\rangle)/\sqrt{2}$. The two basis weights are *fixed* at $1/\sqrt{2}$; only the phase enters, through the relative phase of the $\lvert 1\rangle$ component. It keeps the phase and **nothing else** — in particular it is blind to amplitude.
- **Amplitude–phase qubit** (`amplitude_phase_state`): $\;\cos\alpha\,\lvert 0\rangle + \sin\alpha\,e^{i\theta}\lvert 1\rangle$, where the polar angle $\alpha = \tfrac{\pi}{2}\,a$ carries the amplitude envelope. Now the $\lvert 1\rangle$ weight *tracks band power*, so a flat-power and a bursty signal with identical phases become **different** states.
- **Covariance density** (`covariance_density`): the trace-normalised second moment of the unit vector $(\cos\theta, \sin\theta)$, a single $2\times 2$ *mixed* state for the whole record. It forgets per-sample phase and keeps the **dispersion** of the phase distribution — how concentrated the phases are, and about which direction.
- **Multi-frequency qutrit** (`multifreq_state`): $\;\tfrac{1}{\sqrt{3}}\bigl(\lvert 0\rangle + e^{i\theta}\lvert 1\rangle + e^{2i\theta}\lvert 2\rangle\bigr)$. It stacks the phase *and its second harmonic*, so it carries the **first and second** circular moments of the phase — structure the single-harmonic qubit cannot represent.

We build all four on the toy signal and confirm each is a legitimate quantum state.

In [ ]:
# A toy signal: a drifting phase with an amplitude envelope in [0, 1].
n_samples = 600
t = np.linspace(0.0, 1.0, n_samples, endpoint=False)
carrier_hz = 8.0
theta = 2 * np.pi * carrier_hz * t + 0.4 * np.sin(2 * np.pi * 1.5 * t)  # phase drifts
amp = 0.5 + 0.5 * np.sin(2 * np.pi * 2.0 * t)                          # envelope in [0, 1]

# Build each embedding on the same signal.
psi_naive = phase_qubit_state(theta)                       # (n, 2) pure qubits
psi_ampphase = amplitude_phase_state(theta, amp)           # (n, 2) pure qubits
rho_cov = covariance_density(theta)                        # (2, 2) one mixed state
psi_multifreq = multifreq_state(theta, harmonics=(1, 2))   # (n, 3) pure qutrits

# Confirm each is a valid state.
def per_sample_norms(psi):
    return np.sum(np.abs(psi) ** 2, axis=1)

cov_eigs = np.linalg.eigvalsh(rho_cov.real)
print('phase qubit       shape', psi_naive.shape,
      ' all rows unit-norm:', bool(np.allclose(per_sample_norms(psi_naive), 1.0)))
print('amplitude-phase   shape', psi_ampphase.shape,
      ' all rows unit-norm:', bool(np.allclose(per_sample_norms(psi_ampphase), 1.0)))
print('multi-frequency   shape', psi_multifreq.shape,
      ' all rows unit-norm:', bool(np.allclose(per_sample_norms(psi_multifreq), 1.0)))
print('covariance        shape', rho_cov.shape,
      ' trace:', float(np.trace(rho_cov).real),
      ' min eigenvalue:', float(cov_eigs.min()),
      ' PSD:', bool(cov_eigs.min() > -1e-12))

# The defining contrast between the two qubits: the |1>-component weight.
weight_one_naive = np.abs(psi_naive[:, 1]) ** 2          # fixed at 1/2
weight_one_ampphase = np.abs(psi_ampphase[:, 1]) ** 2    # tracks amplitude
print()
print('phase qubit |1>-weight: constant at', float(weight_one_naive[0]),
      '(amplitude-blind)')
print('amplitude-phase |1>-weight ranges over',
      float(weight_one_ampphase.min()), 'to', float(weight_one_ampphase.max()),
      '(tracks band power)')

**Read the output.** Every embedding passes the validity check: the three per-sample embeddings (phase qubit, amplitude–phase, multi-frequency) have every row unit-norm, so each sample is a legitimate pure state, and the covariance density is unit-trace with a non-negative spectrum, so it is a legitimate (mixed) density matrix. The shapes tell you their reach: the qubits live in dimension $2$, the multi-frequency state in dimension $3$ (one extra slot for the second harmonic), and the covariance is a single $2\times 2$ matrix summarising the whole record rather than one state per sample.

The last two lines are the heart of the menu. The phase qubit's $\lvert 1\rangle$-weight is pinned at $1/2$ for every sample — it does not move, because the naive embedding fixes both basis weights and lets *only* the phase vary. The amplitude–phase qubit's $\lvert 1\rangle$-weight instead sweeps across its full range as the envelope rises and falls: this embedding lets band power into the state. That single difference is what "what each preserves" means in practice — same signal, different state, because the embedding made a different choice about what to keep.

In [ ]:
# Visualise what each embedding keeps, on the toy signal.
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# (a) The two qubits' |1>-weight over time: fixed vs amplitude-tracking.
ax = axes[0, 0]
ax.plot(t, weight_one_naive, color=COLORS['quantum'], lw=2.2,
        label=r'phase qubit  $|\langle 1|\psi\rangle|^2$')
ax.plot(t, weight_one_ampphase, color=COLORS['highlight'], lw=2.2,
        label=r'amplitude-phase  $|\langle 1|\psi\rangle|^2$')
ax.plot(t, amp, color=COLORS['muted'], lw=1.4, ls='--', label='amplitude envelope $a(t)$')
ax.set_xlabel('time  $t$')
ax.set_ylabel(r'$|1\rangle$-component weight')
ax.set_title('Phase qubit is amplitude-blind; amplitude-phase tracks power', pad=10)
ax.legend(loc='upper right')

# (b) Covariance density spectrum for concentrated vs spread phases.
rng_demo = np.random.default_rng(3)
spreads = [0.3, 0.7, 1.2, 2.0]
centre = np.pi / 4
gaps = []
for sd in spreads:
    th = centre + sd * rng_demo.standard_normal(4000)
    e = np.sort(np.linalg.eigvalsh(covariance_density(th).real))[::-1]
    gaps.append(e[0] - e[1])
ax = axes[0, 1]
ax.plot(spreads, gaps, marker='o', color=COLORS['source'], lw=2.2, markersize=8)
ax.set_xlabel('phase spread (std. dev. of $\\theta$, radians)')
ax.set_ylabel('covariance eigenvalue gap')
ax.set_title('Covariance density: anisotropy measures phase concentration', pad=10)

# (c) The toy phase signal itself (wrapped), for context.
ax = axes[1, 0]
ax.plot(t, np.mod(theta, 2 * np.pi), color=COLORS['flow'], lw=1.8)
ax.set_xlabel('time  $t$')
ax.set_ylabel(r'phase  $\theta(t)\ \mathrm{mod}\ 2\pi$')
ax.set_title('The toy signal: a drifting instantaneous phase', pad=10)

# (d) Multi-frequency populations: equal weight across DC, 1st, 2nd harmonic.
ax = axes[1, 1]
populations = np.mean(np.abs(psi_multifreq) ** 2, axis=0)
labels = [r'$|0\rangle$ (DC)', r'$|1\rangle$ (1st harmonic)', r'$|2\rangle$ (2nd harmonic)']
bar_colors = [COLORS['muted'], COLORS['quantum'], COLORS['highlight']]
ax.bar(labels, populations, color=bar_colors, alpha=0.9)
ax.set_ylabel('time-averaged population')
ax.set_ylim(0, 0.5)
ax.set_title('Multi-frequency qutrit: one slot per harmonic', pad=10)

fig.tight_layout()
plt.show()

**Read the figure.** Four panels, one per facet of the menu.

*Top-left* contrasts the two qubits directly. The sky-blue line — the phase qubit's $\lvert 1\rangle$-weight — is flat at $1/2$ no matter what the amplitude does; the rose line — the amplitude–phase qubit's $\lvert 1\rangle$-weight — rises and falls in lockstep with the dashed envelope. Same phase, same signal: the only thing that changed is whether the embedding let amplitude in.

*Top-right* shows what the covariance density keeps. As the phases grow more spread out (moving right along the axis), the gap between the density's two eigenvalues shrinks toward zero: tightly concentrated phases give an anisotropic, near-rank-one state, while phases smeared around the circle give the near-isotropic $\tfrac{1}{2}I$. The covariance embedding does not keep individual phases at all — it keeps this one number, the concentration.

*Bottom-left* is the toy phase itself, wrapped into $[0, 2\pi)$, so you can see the drifting signal all four embeddings are reading.

*Bottom-right* shows the multi-frequency qutrit's three slots carrying equal time-averaged population — the DC reference, the first harmonic, and the second harmonic. The equal weighting is deliberate: each circular moment contributes on the same footing. The extra slot beyond the qubit is exactly where the second-moment structure of the next section will live.

## 3. The key contrast — same first moment, different second moment

Now the contrast that motivates the whole capstone, kept deliberately small. We build **two** phase-difference signals engineered to be indistinguishable to PLV yet distinguishable by a richer embedding, and watch each embedding react.

First, the quantities. Write $\delta = \theta_A - \theta_B$ for the phase difference. Its **circular moments** (Mardia & Jupp 2000) are the expectations $\langle e^{i\delta}\rangle$ (first), $\langle e^{2i\delta}\rangle$ (second), and so on. Two facts connect them to what we have built:

- **PLV is the magnitude of the first circular moment**: $\mathrm{PLV} = \lvert\langle e^{i\delta}\rangle\rvert$ (Lachaux et al. 1999). And the naive joint density's relevant coherence is proportional to it — for the phase qubit, the off-diagonal element $\rho[1, 2]$ (the $\lvert 0\rangle_A\lvert 1\rangle_B$ versus $\lvert 1\rangle_A\lvert 0\rangle_B$ coherence) equals $\tfrac{1}{4}\langle e^{-i\delta}\rangle$, so its magnitude is exactly $\mathrm{PLV}/4$. **The naive embedding is PLV-equivalent by construction.**
- The **multi-frequency** joint density additionally exposes the *second* moment: its element $\rho[6, 2]$ (the $\lvert 2\rangle_A\lvert 0\rangle_B$ versus $\lvert 0\rangle_A\lvert 2\rangle_B$ coherence) equals $\tfrac{1}{9}\langle e^{2i\delta}\rangle$, the second circular moment PLV never touches. (The first moment lives separately, at $\rho[3, 1]$.)

So if we make two signals with the **same** first moment but **different** second moment, the naive coherence (and PLV) must match, while the multi-frequency second-moment coherence must differ. We construct them with a transparent two-population recipe. Let $\theta_A$ sweep the circle uniformly (a phase-symmetric reference, so only the *difference* moments survive). Then assign the per-sample difference $\delta$ to fixed-fraction populations:

- **Signal 1** — an equal split between $\delta = +s_1$ and $\delta = -s_1$. This is symmetric, so its first moment is real and equals $\cos s_1$; its second moment is $\cos 2s_1$.
- **Signal 2** — a fraction $w$ of samples at $\delta = 0$ mixed with an equal split between $\delta = +s_2$ and $\delta = -s_2$. Its first moment is $w + (1-w)\cos s_2$ and its second is $w + (1-w)\cos 2s_2$.

Choosing $w$ to make signal 2's first moment equal signal 1's leaves the two second moments free to differ. We pick the numbers, build the phases, and read every quantity off.

In [ ]:
# Two phase-difference signals: matched FIRST circular moment, distinct SECOND.
# theta_A sweeps the circle uniformly -> a phase-symmetric reference, so only the
# DIFFERENCE moments survive in the joint coherences.
n = 180_000
theta_a = np.linspace(0.0, 2 * np.pi, n, endpoint=False)


def circular_moments(weight_zero, spread):
    """First and second circular moments of the two-population difference recipe."""
    first = weight_zero + (1.0 - weight_zero) * np.cos(spread)
    second = weight_zero + (1.0 - weight_zero) * np.cos(2.0 * spread)
    return first, second


# Signal 1: pure symmetric two-point split at +/- s1 (no spike).
s1 = 0.6
first_target, second_1 = circular_moments(0.0, s1)

# Signal 2: choose s2, then solve the spike weight w2 so the FIRST moment matches.
s2 = 1.2
w2 = (first_target - np.cos(s2)) / (1.0 - np.cos(s2))
first_2, second_2 = circular_moments(w2, s2)

print('design (analytic targets):')
print(f'  signal 1:  first moment = {first_target:.4f}   second moment = {second_1:.4f}')
print(f'  signal 2:  first moment = {first_2:.4f}   second moment = {second_2:.4f}   (spike weight w = {w2:.4f})')
print(f'  first moments matched: {bool(np.isclose(first_target, first_2))}    second moments distinct: {bool(not np.isclose(second_1, second_2))}')


def difference_signal(weight_zero, spread, n_total):
    """Deterministic per-sample phase difference via exact fixed-fraction populations."""
    delta = np.empty(n_total)
    n_zero = int(round(weight_zero * n_total))
    delta[:n_zero] = 0.0
    n_rest = n_total - n_zero
    n_plus = n_rest // 2
    delta[n_zero:n_zero + n_plus] = spread
    delta[n_zero + n_plus:] = -spread
    return delta


delta_1 = difference_signal(0.0, s1, n)
delta_2 = difference_signal(w2, s2, n)
theta_b1 = theta_a - delta_1
theta_b2 = theta_a - delta_2

In [ ]:
# Read every quantity off both signals.
# Classical baseline: PLV (the magnitude of the first circular moment).
plv_1 = plv(theta_a, theta_b1)
plv_2 = plv(theta_a, theta_b2)

# Naive phase-qubit joint density: the PLV-level coherence is rho[1, 2] (= <e^{-i delta}>/4).
rho_naive_1 = joint_density_matrix(theta_a, theta_b1)
rho_naive_2 = joint_density_matrix(theta_a, theta_b2)
naive_coh_1 = abs(rho_naive_1[1, 2])
naive_coh_2 = abs(rho_naive_2[1, 2])

# Multi-frequency qutrit joint density: first moment at rho[3, 1], second at rho[6, 2].
rho_mf_1 = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b1))
rho_mf_2 = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b2))
mf_first_1, mf_second_1 = abs(rho_mf_1[3, 1]), abs(rho_mf_1[6, 2])
mf_first_2, mf_second_2 = abs(rho_mf_2[3, 1]), abs(rho_mf_2[6, 2])

header = f"{'quantity':<44}{'signal 1':>12}{'signal 2':>12}"
print(header)
print('-' * len(header))
print(f"{'PLV  |<e^{i delta}>|  (classical baseline)':<44}{plv_1:>12.4f}{plv_2:>12.4f}")
print(f"{'naive coherence  |rho[1,2]|  (PLV-level)':<44}{naive_coh_1:>12.4f}{naive_coh_2:>12.4f}")
print(f"{'multifreq 1st moment  |rho[3,1]|':<44}{mf_first_1:>12.4f}{mf_first_2:>12.4f}")
print(f"{'multifreq 2nd moment  |rho[6,2]|':<44}{mf_second_1:>12.4f}{mf_second_2:>12.4f}")

# An optional coupling readout: quantum mutual information of the multi-frequency joint state.
qmi_1 = quantum_mutual_information(rho_mf_1, dims=[3, 3], base=np.e)
qmi_2 = quantum_mutual_information(rho_mf_2, dims=[3, 3], base=np.e)
print()
print(f'multi-frequency QMI (nats):  signal 1 = {qmi_1:.4f}   signal 2 = {qmi_2:.4f}')

**Read the output.** Read the table top to bottom. The first two rows — **PLV** and the **naive coherence** $\lvert\rho[1,2]\rvert$ — print the *same* value for both signals (the naive coherence is exactly $\mathrm{PLV}/4$, so it matches to every printed digit). By construction the two signals share their first circular moment, and the naive embedding sees *only* the first moment, so to PLV and to the phase qubit these two signals are **identical**. A discriminating analysis built on either one cannot tell them apart.

The third row confirms the multi-frequency embedding's *first*-moment coherence $\lvert\rho[3,1]\rvert$ also matches — as it must, since it too is fixed by the shared first moment. The decisive row is the fourth: the multi-frequency *second*-moment coherence $\lvert\rho[6,2]\rvert$ is **different** for the two signals. The second harmonic slot carries $\langle e^{2i\delta}\rangle$, and we engineered the two second moments to differ — so the qutrit state genuinely differs between signal 1 and signal 2 in a place the qubit does not even have. The quantum mutual information readout follows suit: the multi-frequency joint states carry different amounts of coupling, because they *are* different states.

The reading is exactly the punchline of the notebook: **the naive embedding is PLV-equivalent by construction — it cannot, in principle, separate these two signals — while a richer embedding sees more.** This is not yet a demonstration that QOT beats PLV on a real coupling problem; it is the structural precondition that makes such a demonstration possible. We make the contrast visible in one figure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (a) The two difference distributions: same first moment, different shape.
ax = axes[0]
bins = np.linspace(-np.pi, np.pi, 60)
ax.hist(np.mod(delta_1 + np.pi, 2 * np.pi) - np.pi, bins=bins, density=True,
        color=COLORS['quantum'], alpha=0.7, label='signal 1')
ax.hist(np.mod(delta_2 + np.pi, 2 * np.pi) - np.pi, bins=bins, density=True,
        color=COLORS['highlight'], alpha=0.6, label='signal 2')
ax.set_xlabel(r'phase difference  $\delta = \theta_A - \theta_B$  (radians)')
ax.set_ylabel('density')
ax.set_title('Two difference signals, matched first circular moment', pad=10)
ax.legend()

# (b) The decisive comparison: what each embedding sees.
ax = axes[1]
groups = ['PLV\n(baseline)', 'naive\ncoherence', 'multifreq\n1st moment', 'multifreq\n2nd moment']
vals_1 = [plv_1, naive_coh_1, mf_first_1, mf_second_1]
vals_2 = [plv_2, naive_coh_2, mf_first_2, mf_second_2]
x = np.arange(len(groups))
ax.bar(x - 0.19, vals_1, width=0.38, color=COLORS['quantum'], alpha=0.9, label='signal 1')
ax.bar(x + 0.19, vals_2, width=0.38, color=COLORS['highlight'], alpha=0.9, label='signal 2')
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=9)
ax.set_ylabel('magnitude')
ax.set_title('Matched everywhere PLV looks; split only at the 2nd moment', pad=10)
ax.legend()

fig.tight_layout()
plt.show()

**Read the figure.** The left panel shows the two engineered phase-difference distributions. They are visibly *different shapes* — signal 1 is a clean symmetric pair of spikes, signal 2 adds a central spike at $\delta = 0$ — yet they were built to share the same first circular moment. Different distributions can hide behind the same first moment, and that is precisely the blind spot we are probing.

The right panel is the verdict, four grouped pairs of bars. For **PLV**, the **naive coherence**, and the **multi-frequency first moment**, the sky-blue (signal 1) and rose (signal 2) bars are the *same height* — every quantity that depends only on the first circular moment cannot tell the two signals apart. Only the last pair, the **multi-frequency second moment**, shows a visible gap between the bars. That single split is the whole story: PLV and the naive embedding are flat against this difference, and only the embedding that keeps the second moment registers it. Choosing the embedding is choosing whether your geometry can see this gap at all.

## Your turn

**Warm-up.** In the menu of section 2, the phase qubit and the amplitude–phase qubit were built on the *same* toy signal. Picture two signals with identical phase traces but very different amplitude envelopes — one with flat power, one with strong bursts. Which of the two embeddings would give *identical* states for this pair, and which would give *different* states? Name the single feature of each embedding that decides the answer.

**Go further.** The contrast in section 3 was engineered so the two signals share their *first* circular moment while differing in the *second*. Describe in words how you would instead engineer two signals that share *both* the first and the second circular moment but differ in the *third*. Which embedding on the menu would then be needed to tell them apart — and what would you change in the `multifreq_state` call to give the joint density a slot for that third moment?

**Challenge.** The naive coherence was shown to equal $\mathrm{PLV}/4$, and the multi-frequency second-moment coherence to equal $\langle e^{2i\delta}\rangle / 9$ in magnitude. Both carry a constant prefactor ($1/4$ for the qubit, $1/9$ for the qutrit). Explain where each prefactor comes from — relate it to the per-sample normalisation of the embedding (the $1/\sqrt{2}$ and $1/\sqrt{3}$ in the state definitions) and to how two such factors combine when the joint density is formed from a product of two states. Then argue why these prefactors, being the *same* for both signals, leave every "same versus different" conclusion of section 3 untouched.

## What you built

- You established that the embedding is a **modelling decision**: the QOT objects from `09` are fixed, so the number they return is decided by the state you build — and the state is built by an embedding that *chooses* what structure of the signal to keep.
- You built the **four embeddings** of `qot_course.quantum_ot` on one toy signal — the phase qubit, the amplitude–phase qubit, the covariance density, and the multi-frequency qutrit — and confirmed each is a valid quantum state, with shapes and weights that make their differences concrete.
- You named **what each preserves**: phase only (the naive qubit), phase and band power (amplitude–phase), phase dispersion (covariance), and the first *and* second circular moments (multi-frequency).
- You showed, on a clean two-population toy, that two phase-difference signals with the **same first circular moment** give the naive embedding the *same* PLV-level coherence — it is PLV-equivalent by construction — while the **multi-frequency** embedding's second-moment coherence $\lvert\rho[6,2]\rvert$ *differs*. A richer embedding sees structure PLV cannot.

Stand back and see what this gives you. You can now choose an embedding *deliberately*, knowing exactly what each one lets the geometry see — and you have proven, in miniature, that the choice can decide whether a piece of structure is visible at all. That is the bridge from machinery to data, and it is the precise hinge the capstone turns on.

## What's next

In `11_kuramoto_dyad_and_ground_truth` you will build a synthetic dyad with a *known* coupling strength — the ground truth against which every measure is judged. Across `11`–`15` the embedding choice you learned to make here decides whether quantum optimal transport can separate signals that share a PLV, turning today's small toy contrast into the capstone's full discriminating experiment.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091

**Previous:** `notebooks/04_QuantumOptimalTransport/09_the_qot_zoo.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/11_kuramoto_dyad_and_ground_truth.ipynb`